# Usage modules

## Divergence Module

In [1]:
import numpy as np
from kfc_procedure.core.clustering.divergences.base import (
    BregmanDivergenceFactory, 
    BaseBregmanDivergence
)

# list available divergences
print(f"Divergances : {BregmanDivergenceFactory.available()}")

Divergances : ['euclidean', 'gkl', 'is', 'logistic']


In [2]:
# create a euclidean divergence instance using factory
div = BregmanDivergenceFactory.create("euclidean")
print(f"Created divergence : {div}")

Created divergence : SquaredEuclidean {'validate_domain': True})


In [3]:
# compute divergence between two points
X = np.array([
    [1.0, 2.0],
    [2.0, 3.0],
    [10.0, 10.0]
])

Y = np.array([
    [1.0, 1.0],
    [9.0, 9.0]
])
d = div.distance(X, Y)
print(f"Divergence between X and Y :\n{d}")
print(f"Shape of divergence matrix : {d.shape}")

Divergence between X and Y :
[[  1. 113.]
 [  5.  85.]
 [162.   2.]]
Shape of divergence matrix : (3, 2)


In [4]:
labels = div.assign_clusters(X, Y)
center = div.centroid(X)
print(f"Assigned cluster labels : {labels}")
print(f"Centroid of X : {center}")

Assigned cluster labels : [0 0 1]
Centroid of X : [4.33333333 5.        ]


In [5]:
# create custom divergence
if "mahalanobis" not in BregmanDivergenceFactory.available():
    @BregmanDivergenceFactory.register("mahalanobis")
    class MahalanobisDivergence(BaseBregmanDivergence):
        """
        where A is a symmetric positive definite (SPD) matrix
        - in domain: R^d
        - φ(x) = 1/2 x^T A x
        - D_φ(x, y) = φ(x) - φ(y) - <∇φ(y), x - y>
                    = 1/2 (x - y)^T A (x - y)
        """
        name = "mahalanobis"
        family = "gaussian"

        def __init__(
            self,
            A: np.ndarray | None = None,
            validate_domain: bool = True
        ):
            super().__init__(validate_domain=validate_domain)
            self.A = A
        
        def in_domain(self, X: np.ndarray) -> bool:
            """
            Domain is all real vectors R^d.
            """
            return np.isfinite(X).all()
        
        def phi(self, X):
            d = X.shape[1]
            if self.A is None:
                A = np.eye(d)
            else:
                A = self.A
            
            if A.shape != (d, d):
                raise ValueError(
                    f"matrix must have shape ({d}, {d}), got {A.shape}"
                )

            if not np.allclose(A, A.T):
                raise ValueError("matrix must be symmetric")

            return 0.5 * np.einsum("ij,jk,ik->i", X, A, X)
        
        def grad_phi(self, X):
            X = np.asarray(X, dtype=np.float64)
            d = X.shape[1]
            if self.A is None:
                A = np.eye(d)
            else:
                A = self.A
            return X @ A
    

In [6]:
div = BregmanDivergenceFactory.create("mahalanobis", A=np.array([[1, 0], [0, 2]]))
d = div.distance(X, Y)
print(f"Mahalanobis divergence between X and Y :\n{d}")

Mahalanobis divergence between X and Y :
[[  1.   81. ]
 [  4.5  60.5]
 [121.5   1.5]]


In [7]:
# usage kmeanbregman
from kfc_procedure.core.clustering.bregman import BregmanKMeans

kmean = BregmanKMeans(
    n_clusters=2,
    divergence="mahalanobis",
    divergence_params={"A": np.array([[1, 0], [0, 2]])}
)
kmean.fit(X)
print(f"Cluster centers :\n{kmean.cluster_centers_}")
print(f"Cluster labels : {kmean.labels_}")


Cluster centers :
[[10.  10. ]
 [ 1.5  2.5]]
Cluster labels : [1 1 0]


/Users/ougi/Documents/Project/kfc-procedure/src/kfc_procedure/core/clustering/bregman.py:209: RuntimeWarning: invalid value encountered in scalar divide
  change = abs(prev - dist) / (abs(prev) + 1e-12)


## Local Models Module

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes, load_iris

X_reg, y_reg = load_diabetes(return_X_y=True)
X_clf, y_clf = load_iris(return_X_y=True)

# split into train/test
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)
X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42
)

print(
    f"Regression dataset: X={X_reg.shape}, y={y_reg.shape}\n"
    f"Classification dataset: X={X_clf.shape}, y={y_clf.shape}"
)

Regression dataset: X=(442, 10), y=(442,)
Classification dataset: X=(150, 4), y=(150,)


In [9]:
from kfc_procedure.core.ml.base import (
    BaseLocalModelClassifier,
    BaseLocalModelRegressor,
    LocalModelClassifierFactory,
    LocalModelRegressorFactory
)

print(f"Available local regression models: {LocalModelRegressorFactory.available()}")
print(f"Available local classification models: {LocalModelClassifierFactory.available()}")

Available local regression models: ['decision_tree', 'decision_tree_regression', 'dtr', 'lasso', 'lasso_regression', 'linear', 'linear_regression', 'ols', 'random_forest', 'random_forest_regression', 'rf', 'ridge', 'ridge_regression', 'rr']
Available local classification models: ['decision_tree', 'dt', 'logistic', 'logistic_regression', 'lr', 'random_forest', 'rf']


In [10]:
# use linear regression
lr = LocalModelRegressorFactory.create("linear")
lr.fit(X_reg_train, y_reg_train)
y_reg_pred = lr.predict(X_reg_test)
mse = np.mean((y_reg_pred - y_reg_test) ** 2)
print(f"Linear Regression MSE: {mse}")

Linear Regression MSE: 2900.1936284934804


In [11]:
# use logistic regression
logr = LocalModelClassifierFactory.create("logistic")
logr.fit(X_clf_train, y_clf_train)
y_clf_pred = logr.predict(X_clf_test)
accuracy = np.mean(y_clf_pred == y_clf_test)
print(f"Logistic Regression Accuracy: {accuracy}")

Logistic Regression Accuracy: 1.0


In [12]:
# create a custom local model for regression
from sklearn.svm import SVR

if "svr" not in LocalModelRegressorFactory.available():
    @LocalModelRegressorFactory.register("svr")
    class SVRLocalModel(BaseLocalModelRegressor):
        def __init__(self, **kwargs):
            super().__init__()
            self.model = SVR(**kwargs)
        
        def fit(self, X, y):
            self.model.fit(X, y)
        
        def predict(self, X):
            return self.model.predict(X)
    
svr = LocalModelRegressorFactory.create("svr", kernel="rbf", C=1.0)
svr.fit(X_reg_train, y_reg_train)
y_svr_pred = svr.predict(X_reg_test)
mse_svr = np.mean((y_svr_pred - y_reg_test) ** 2)
print(f"SVR Local Model MSE: {mse_svr}")

SVR Local Model MSE: 4333.285954518086


In [13]:
# create a custom local model for classification
from sklearn.ensemble import RandomForestClassifier

if "rfc" not in LocalModelClassifierFactory.available():
    @LocalModelClassifierFactory.register("rfc")
    class RFCLocalModel(BaseLocalModelClassifier):
        def __init__(self, **kwargs):
            super().__init__()
            self.model = RandomForestClassifier(**kwargs)
        
        def fit(self, X, y):
            self.model.fit(X, y)
        
        def predict(self, X):
            return self.model.predict(X)
        
        def predict_proba(self, X):
            return self.model.predict_proba(X)

rfc = LocalModelClassifierFactory.create("rfc", n_estimators=100, random_state=42)
rfc.fit(X_clf_train, y_clf_train)
y_rfc_pred = rfc.predict(X_clf_test)
accuracy_rfc = np.mean(y_rfc_pred == y_clf_test)
print(f"Random Forest Classifier Local Model Accuracy: {accuracy_rfc}")


Random Forest Classifier Local Model Accuracy: 1.0


## Aggregations

In [14]:
from kfc_procedure.core.aggregations.base import (
    AggregationRegressorFactory,
    AggregationClassifierFactory,
    BaseAggregationClassifier,
    BaseAggregationRegressor
)

print(f"Available aggregation regressors: {AggregationRegressorFactory.available()}")
print(f"Available aggregation classifiers: {AggregationClassifierFactory.available()}")

Available aggregation regressors: ['gradientcobra', 'mean', 'stacking', 'weighted_mean']
Available aggregation classifiers: ['cc', 'combine_classifier', 'majority_vote', 'stacking']


In [15]:
# create a instance of mean regressor
mean_agg = AggregationRegressorFactory.create("mean")
mean_agg.fit(predictions=X_reg_train, y=y_reg_train)
y_mean_pred = mean_agg.predict(X_reg_test)
mse_mean = np.mean((y_mean_pred - y_reg_test) ** 2)
print(f"Mean Aggregation Regressor MSE: {mse_mean}")


Mean Aggregation Regressor MSE: 26547.372321567702


In [16]:
# majority_vote
majority_agg = AggregationClassifierFactory.create("majority_vote")
majority_agg.fit(predictions=X_clf_train, y=y_clf_train)
y_majority_pred = majority_agg.predict(X_clf_test)
accuracy_majority = np.mean(y_majority_pred == y_clf_test)
print(f"Majority Vote Aggregation Classifier Accuracy: {accuracy_majority}")


Majority Vote Aggregation Classifier Accuracy: 0.9333333333333333


In [17]:
# create a custom aggregation model for regression
from sklearn.ensemble import RandomForestRegressor

if "rfr" not in AggregationRegressorFactory.available():
    @AggregationRegressorFactory.register("rfr")
    class RFRAggregationModel(BaseAggregationRegressor):
        def __init__(self, **kwargs):
            super().__init__()
            self.model = RandomForestRegressor(**kwargs)
        
        def fit(self, predictions, y):
            self.model.fit(predictions, y)
        
        def predict(self, predictions):
            return self.model.predict(predictions)
    
rfr_agg = AggregationRegressorFactory.create("rfr", n_estimators=100, random_state=42)
rfr_agg.fit(predictions=X_reg_train, y=y_reg_train)
y_rfr_pred = rfr_agg.predict(X_reg_test)
mse_rfr = np.mean((y_rfr_pred - y_reg_test) ** 2)
print(f"Random Forest Regressor Aggregation Model MSE: {mse_rfr}")

Random Forest Regressor Aggregation Model MSE: 2952.0105887640448


In [18]:
# create a custom aggregation model for classification
from sklearn.ensemble import RandomForestClassifier
if "rfr" not in AggregationClassifierFactory.available():
    @AggregationClassifierFactory.register("rfc")
    class RFCAggregationModel(BaseAggregationClassifier):
        def __init__(self, **kwargs):
            super().__init__()
            self.model = RandomForestClassifier(**kwargs)
        
        def fit(self, predictions, y):
            self.model.fit(predictions, y)
        
        def predict(self, predictions):
            return self.model.predict(predictions)
        
        def predict_proba(self, predictions):
            return self.model.predict_proba(predictions)

rfc_agg = AggregationClassifierFactory.create("rfc", n_estimators=100, random_state=42)
rfc_agg.fit(predictions=X_clf_train, y=y_clf_train)
y_rfc_agg_pred = rfc_agg.predict(X_clf_test)
accuracy_rfc_agg = np.mean(y_rfc_agg_pred == y_clf_test)
print(f"Random Forest Classifier Aggregation Model Accuracy: {accuracy_rfc_agg}")

Random Forest Classifier Aggregation Model Accuracy: 1.0


## K Step

In [22]:
from kfc_procedure.core.steps.kstep import KStep, BaseKStep

kstep = KStep(
    divergences=[
        {
            'name' : 'euclidean',
        },
        {
            'name' : 'mahalanobis',
            'params' : {
                'divergence_params' : {
                    'A' : np.eye(10)
                },
                "n_init" : 100
            }
        },
    ]
)
kstep.fit(X_reg_train)

/Users/ougi/Documents/Project/kfc-procedure/src/kfc_procedure/core/clustering/bregman.py:209: RuntimeWarning: invalid value encountered in scalar divide
  change = abs(prev - dist) / (abs(prev) + 1e-12)
/Users/ougi/Documents/Project/kfc-procedure/src/kfc_procedure/core/clustering/bregman.py:209: RuntimeWarning: invalid value encountered in scalar divide
  change = abs(prev - dist) / (abs(prev) + 1e-12)


,divergences,"[{'name': 'euclidean'}, {'name': 'mahalanobis', 'params': {'divergence_params': {'A': array([[1., 0... 0., 0., 1.]])}, 'n_init': 100}}]"


In [23]:
print(f"Cluster centers after KStep: \n{kstep.clusters_}")

Cluster centers after KStep: 
{'DB0': array([1, 1, 6, 3, 3, 3, 6, 0, 7, 3, 2, 0, 5, 0, 3, 1, 3, 5, 5, 6, 5, 1,
       6, 6, 6, 0, 7, 7, 5, 3, 7, 7, 7, 5, 1, 5, 5, 6, 5, 1, 3, 1, 2, 0,
       0, 6, 1, 0, 1, 3, 6, 3, 4, 4, 3, 4, 4, 6, 4, 4, 1, 1, 7, 2, 4, 5,
       1, 3, 5, 1, 1, 5, 3, 5, 2, 2, 1, 2, 3, 2, 5, 5, 3, 5, 6, 0, 0, 6,
       6, 0, 1, 2, 0, 1, 3, 5, 1, 3, 7, 6, 6, 3, 7, 1, 1, 4, 3, 0, 4, 1,
       7, 2, 6, 4, 3, 2, 1, 0, 3, 7, 5, 2, 6, 3, 5, 4, 2, 0, 2, 2, 6, 5,
       6, 3, 5, 0, 7, 1, 4, 6, 1, 1, 5, 6, 0, 0, 6, 1, 2, 4, 5, 1, 7, 2,
       3, 7, 3, 0, 3, 6, 0, 0, 4, 3, 1, 3, 7, 3, 1, 7, 7, 5, 5, 0, 3, 2,
       2, 6, 2, 3, 1, 1, 1, 4, 7, 3, 5, 3, 2, 6, 7, 1, 3, 0, 0, 3, 1, 3,
       7, 3, 4, 6, 2, 2, 3, 5, 1, 3, 6, 7, 3, 5, 0, 7, 4, 2, 3, 6, 1, 6,
       0, 3, 2, 7, 3, 6, 3, 1, 2, 3, 5, 3, 0, 1, 3, 0, 6, 2, 0, 7, 5, 4,
       2, 1, 7, 1, 0, 5, 0, 3, 5, 1, 5, 6, 2, 3, 1, 0, 7, 4, 5, 0, 1, 6,
       6, 2, 7, 6, 4, 5, 7, 6, 0, 2, 5, 4, 2, 6, 6, 1, 6, 5, 6, 2, 1, 1,
       6, 7, 

In [25]:
kstep.divergences

[{'name': 'euclidean'},
 {'name': 'mahalanobis',
  'params': {'divergence_params': {'A': array([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
           [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
           [0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
           [0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
           [0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
           [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
           [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
           [0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
           [0., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
           [0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]])},
   'n_init': 100}}]

## F Step

In [27]:
from kfc_procedure.core.steps.fstep import FStep, BaseFStep

models = FStep(
    config={
        'name' : 'linear',
    }
)
models.fit(X_reg_train, y_reg_train, clusters=kstep.clusters_)
print(f"Local models after FStep: \n{models.models_}")


Local models after FStep: 
{'DB0': {np.int64(0): LinearRegression(), np.int64(1): LinearRegression(), np.int64(2): LinearRegression(), np.int64(3): LinearRegression(), np.int64(4): LinearRegression(), np.int64(5): LinearRegression(), np.int64(6): LinearRegression(), np.int64(7): LinearRegression()}, 'DB1': {np.int64(0): LinearRegression(), np.int64(1): LinearRegression(), np.int64(2): LinearRegression(), np.int64(3): LinearRegression(), np.int64(4): LinearRegression(), np.int64(5): LinearRegression(), np.int64(6): LinearRegression(), np.int64(7): LinearRegression()}}


## C Step

In [29]:
from kfc_procedure.core.steps.cstep import CStep, BaseCStep

cstep = CStep(
    config={
        'name' : 'mean',
    }
)
# build predictions from train data
predictions = models.predict(X_reg_train, clusters=kstep.clusters_)
cstep.fit(predictions=predictions, y=y_reg_train)

,config,{'name': 'mean'}


In [40]:
preds = np.hstack(list(predictions.values()))
y_cstep_pred = cstep.predict(preds)
mse_cstep = np.mean((y_cstep_pred - y_reg_train) ** 2)
print(f"CStep Aggregation MSE: {mse_cstep}")

CStep Aggregation MSE: 3233.3391377311327


In [45]:
for key, pred in predictions.items():
    print(f"Cluster {key} predictions shape: {pred.shape}")
print(f"preds shape : {preds.shape}")

Cluster DB0 predictions shape: (353, 8)
Cluster DB1 predictions shape: (353, 8)
preds shape : (353, 16)


## KFC Procedure

In [ ]:
from kfc_procedure.kfc import KFCRegressor, KFCClassifier

kfc_reg = KFCRegressor(
    kstep=[
        {
            'name' : 'euclidean',
        },
        {
            'name' : 'mahalanobis',
            'params' : {
                'divergence_params' : {
                    'A' : np.eye(10)
                },
                "n_init" : 100
            }
        },
    ],
    fstep={
        'name' : 'linear',
    },
    cstep={
        'name' : 'mean',
    }
)
kfc_reg.fit(X_reg_train, y_reg_train)
y_kfc_reg_pred = kfc_reg.predict(X_reg_test)
mse_kfc_reg = np.mean((y_kfc_reg_pred - y_reg_test) ** 2)
print(f"KFC Regressor MSE: {mse_kfc_reg}")

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3926325776.py, line 4)